## Silver Transformation – Location County

This notebook creates the Silver table capstone.silver.location_county.

**Dependency:**
Uses the base geography transformation from the _geography_base notebook.

**Source:**
Cleaned geography dataframe geo.

**Transformation logic:**
- Selects standardized location attributes
- Keeps location_id, state_fips, county_fips, state_name, and county_name
- Removes duplicate records by keeping the latest ingestion timestamp

**Output table:**
capstone.silver.location_county

**Purpose:**
Provides a standardized county-level location dataset that can be used by population and other analytical datasets.

In [0]:
 %python
df = spark.table("capstone.bronze.geography_population_raw")
display(df.limit(20))
print(df.columns)

In [0]:
%run ./_geography_base

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window





# 1) Select columns for Silver table
location_county = geo.select(
  F.col("location_id"),
  F.col("state_fips"),
  F.col("county_fips"),
  F.col("state_name"),
  F.col("county_name"),
  F.col("GEO_ID").alias("geo_id_raw"),
  F.col("ingestion_timestamp"),
  F.col("source_system")
)

# 2) Deduplicate (keep latest by ingestion_timestamp)
w = Window.partitionBy("location_id").orderBy(F.col("ingestion_timestamp").desc_nulls_last())

location_county = (
  location_county
  .withColumn("rn", F.row_number().over(w))
  .filter(F.col("rn") == 1)
  .drop("rn")
)

(location_county.write
  .mode("overwrite")
  .format("delta")
  .option("overwriteSchema","true")
  .saveAsTable("capstone.silver.location_county")
)

print("Created: capstone.silver.location_county | rows =", location_county.count())
display(location_county.limit(20))